In [2]:
# imports 

import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
words = open('names.txt').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = { ch: i for i, ch in enumerate(chars) }
stoi['.'] = 0
itos = { i: ch for ch, i in stoi.items() }

In [4]:
# starter code

X, Y = [], []
block_size = 3  # context lenght of how many characters to consider to predict the next one

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y) 
X.shape, Y.shape # dataset

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
w1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
w2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, w1, b1, w2, b2]

for p in parameters:
    # p.requires_grad = True
    p.requires_grad_()



In [5]:
# training loop

for _ in range(10):
    # minibatching
    ix = torch.randint(0, X.shape[0], (32,))
    # forward pass
    emb = C[X[ix]] # 32 x 3 x 2
    h = torch.tanh(emb.view(-1, 6) @ w1 + b1)
    logits = h @ w2 + b2

    # three lines below can be replaced by F.cross_entropy which is much more efficient due to pytorch implementation
    # counts = logits.exp()
    # prob = counts / counts.sum(dim=1, keepdim=True) # softmax
    # loss = -prob[torch.arange(32), Y].log().mean()
    loss = F.cross_entropy(logits, Y[ix])
    print(loss.item())

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data -= p.grad * 0.1 # learning rate 0.1

21.244462966918945
15.64154052734375
15.680699348449707
16.553890228271484
14.796073913574219
11.696124076843262
15.069952964782715
11.959941864013672
13.051692008972168
10.21394157409668


In [6]:
# current problems with MLP implementation


# 1. initial loss
